# MultiIndex and Advanced Indexing

## Quick notes

### What this covers
Hierarchical (MultiIndex) rows, slicing them with `.loc`, `pd.IndexSlice` and `xs`, reordering and sorting levels, `set_index`/`reset_index`, grouping by levels, label alignment in arithmetic, `reindex`/`align`, duplicate labels, `RangeIndex`/`IntervalIndex`, `pd.cut`/`pd.qcut`, and the filtering helpers `isin`, `between`, `query`, `eval`, `where`, `mask`, `filter`. Basic `loc`/`iloc` is in 02, groupby basics in 05, reshaping in 07.

### Key points
- Build a MultiIndex with `from_tuples` (pairs), `from_product` (full grid), `from_arrays` (one list per level) or `from_frame`. Always set `names=`.
- `df.loc["East"]` picks the outer level; `df.loc[("East", "Q2")]` picks one row; `df.loc[pd.IndexSlice[:, "Q1":"Q2"], :]` slices an inner level.
- `df.xs("Q1", level="quarter")` picks by any level without IndexSlice. It drops that level unless `drop_level=False`.
- A MultiIndex stores `.levels` (unique labels) and `.codes` (integer pointers). `get_level_values(name)` returns the full-length labels of one level.
- `swaplevel` / `reorder_levels` change level order but do not sort rows; follow with `sort_index()`.
- `set_index([...])` makes columns into levels; `reset_index()` brings them back (`level=` for one, `drop=True` to discard).
- `groupby(level="region")` groups by an index level directly, no reset needed.
- Arithmetic aligns by LABEL. `.add(other, fill_value=0)` handles one-sided labels.
- `reindex` re-selects rows to match a new label list (missing become NaN, `method="ffill"` fills gaps on sorted index); `reindex_like` copies another object's index; `align` puts two objects on a common index.
- `RangeIndex` is the default and costs almost no memory. Filtering keeps old labels, so `.loc` and `.iloc` start to disagree.
- `pd.cut` makes bins by edges (right-closed by default); `pd.qcut` makes bins with equal counts; `IntervalIndex` supports lookups by value.
- `isin`, `between`, `query("a > @x")`, `eval`, `where` (keep where True), `mask` (replace where True), `filter(like=/regex=/items=)` cover most filtering needs.

### Gotchas (the part worth revising twice)
- Slicing a range on an unsorted MultiIndex raises `UnsortedIndexError`. Sort first with `sort_index()`.
- After filtering rows, `.levels` still lists removed labels. Use `remove_unused_levels()` when you read levels.
- `a + b` on Series with different labels gives NaN for the labels that do not match, silently. Same length does not mean matched by position.
- Duplicate labels make `.loc["a"]` return a Series instead of a scalar, and `reindex` fails. Check `index.is_unique`; guard with `set_flags(allows_duplicate_labels=False)`.
- `pd.cut` with right-closed bins turns the lowest edge value into NaN unless `include_lowest=True`.
- `where` fills with NaN by default, which turns integer columns into floats.
- Use `&`, `|`, `~` with brackets around each condition. `and`/`or` raise "truth value is ambiguous".
- `query` is a mini-language, not full Python; use `@var` for outside variables and backticks for names with spaces.

### When it matters
Any time data has natural hierarchy (region/quarter, country/city, store/product), or when you join, subtract or compare two objects and the result must line up on labels. Understanding alignment and sorted indexes prevents the silent NaN and wrong-row bugs that are hardest to spot.

### One-line recap example
```python
import pandas as pd
s = pd.Series([1, 2, 3, 4], index=pd.MultiIndex.from_product([["E", "W"], ["Q1", "Q2"]], names=["r", "q"]))
s.loc[pd.IndexSlice[:, "Q1"]]  # all regions, only Q1 (index is sorted, so slicing is safe)
```

In [1]:
# 01. Building a MultiIndex
# 02. Slicing a MultiIndex with .loc and pd.IndexSlice
# 03. xs - Cross-Section by Level
# 04. levels, codes, names - What Is Inside
# 05. swaplevel, reorder_levels, sort_index
# 06. set_index and reset_index
# 07. stack and unstack Recap
# 08. groupby on Index Levels
# 09. Index Alignment on Arithmetic (Classic Gotcha)
# 10. reindex, reindex_like, align
# 11. Duplicate Index Problems
# 12. RangeIndex
# 13. IntervalIndex, pd.cut, pd.qcut
# 14. isin and between
# 15. query and eval
# 16. where and mask
# 17. .filter - Select Labels by Pattern
# 18. Boolean Masks Combined With Index Levels
#
# See also: 02-Selection-And-Indexing (loc/iloc basics), 05-GroupBy-And-Aggregation,
# 07-Reshaping (pivot/melt/stack basics).

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

## 01. Building a MultiIndex

- A MultiIndex is an index with 2+ levels: every row label is a tuple.
- from_tuples: you already have a list of (a, b) pairs.
- from_product: all combinations of several lists (a full grid).
- from_arrays: one list per level, same length, matched by position.
- from_frame: take columns of a DataFrame as levels.
- Always give names= so you can refer to levels by name later.

In [2]:
t = pd.MultiIndex.from_tuples([("East", "Q1"), ("East", "Q2"), ("West", "Q1")], names=["region", "quarter"])
print("from_tuples:\n", t, sep="")

p = pd.MultiIndex.from_product([["East", "West"], ["Q1", "Q2", "Q3"]], names=["region", "quarter"])
print("\nfrom_product (2 x 3 = 6 rows):", len(p))

a = pd.MultiIndex.from_arrays([["East", "East", "West"], [1, 2, 1]], names=["region", "n"])
print("from_arrays:", a.tolist())

f = pd.MultiIndex.from_frame(pd.DataFrame({"x": ["a", "b"], "y": [1, 2]}))
print("from_frame names:", f.names)

df = pd.DataFrame({"sales": rng.integers(10, 100, 6), "cost": rng.integers(5, 50, 6)}, index=p)
print("\nDataFrame with MultiIndex rows:\n", df, sep="")

from_tuples:
MultiIndex([('East', 'Q1'),
            ('East', 'Q2'),
            ('West', 'Q1')],
           names=['region', 'quarter'])

from_product (2 x 3 = 6 rows): 6
from_arrays: [('East', 1), ('East', 2), ('West', 1)]
from_frame names: ['x', 'y']

DataFrame with MultiIndex rows:
                sales  cost
region quarter             
East   Q1          18     8
       Q2          79    36
       Q3          68    14
West   Q1          49     9
       Q2          48    28
       Q3          87    48


## 02. Slicing a MultiIndex with .loc and pd.IndexSlice

- df.loc["East"] picks the outer level and drops it from the result.
- df.loc[("East", "Q2")] (a tuple) picks one exact row.
- To slice the INNER level you need pd.IndexSlice: idx[:, "Q1":"Q2"].
- Slicing ranges on a MultiIndex needs it to be SORTED (lexsorted),
  otherwise pandas raises UnsortedIndexError.
- Always pass axis info clearly: df.loc[idx[...], :] is safest.

In [3]:
idx = pd.IndexSlice
print("df.loc['East']:\n", df.loc["East"], sep="")
print("\ndf.loc[('East', 'Q2')]:\n", df.loc[("East", "Q2")], sep="")
print("\ndf.loc[idx[:, 'Q1':'Q2'], :] (inner level range, all regions):\n", df.loc[idx[:, "Q1":"Q2"], :], sep="")
print("\ndf.loc[idx[:, ['Q1', 'Q3']], 'sales'] (list of inner labels):\n", df.loc[idx[:, ["Q1", "Q3"]], "sales"], sep="")

shuffled = df.iloc[[3, 0, 5, 1, 4, 2]]
print("\nIs shuffled index sorted?", shuffled.index.is_monotonic_increasing)
try:
    shuffled.loc[idx["East":"West", "Q1":"Q2"], :]
except Exception as e:
    print("Range slice on unsorted MultiIndex ->", type(e).__name__)
print("After sort_index it works:", shuffled.sort_index().loc[idx[:, "Q1":"Q2"], :].shape)

df.loc['East']:
         sales  cost
quarter             
Q1          18     8
Q2          79    36
Q3          68    14

df.loc[('East', 'Q2')]:
sales    79
cost     36
Name: (East, Q2), dtype: int64

df.loc[idx[:, 'Q1':'Q2'], :] (inner level range, all regions):
                sales  cost
region quarter             
East   Q1          18     8
       Q2          79    36
West   Q1          49     9
       Q2          48    28

df.loc[idx[:, ['Q1', 'Q3']], 'sales'] (list of inner labels):
region  quarter
East    Q1         18
        Q3         68
West    Q1         49
        Q3         87
Name: sales, dtype: int64

Is shuffled index sorted? False
Range slice on unsorted MultiIndex -> UnsortedIndexError
After sort_index it works: (4, 2)


## 03. xs - Cross-Section by Level

- df.xs(key, level=...) selects by one level's value, ANYWHERE in the
  hierarchy, without needing IndexSlice.
- By default xs DROPS the level it matched; drop_level=False keeps it.
- It is a shortcut for the common "give me all Q1 rows" question.

In [4]:
print("df.xs('Q1', level='quarter'):\n", df.xs("Q1", level="quarter"), sep="")
print("\nSame with drop_level=False:\n", df.xs("Q1", level="quarter", drop_level=False), sep="")
print("\nColumn-wise xs on a Series-like: xs(('West','Q2')):\n", df.xs(("West", "Q2")), sep="")

df.xs('Q1', level='quarter'):
        sales  cost
region             
East       18     8
West       49     9

Same with drop_level=False:
                sales  cost
region quarter             
East   Q1          18     8
West   Q1          49     9

Column-wise xs on a Series-like: xs(('West','Q2')):
sales    48
cost     28
Name: (West, Q2), dtype: int64


## 04. levels, codes, names - What Is Inside

- A MultiIndex stores each level as a list of UNIQUE labels (.levels) and
  the rows as integer pointers into them (.codes). This saves memory.
- get_level_values(n or name) gives the full-length labels of one level.
- GOTCHA: .levels can still contain labels that are no longer in the data
  after you filter rows. Use remove_unused_levels() to clean up.

In [5]:
print("levels:", [list(l) for l in df.index.levels])
print("codes: ", [list(c) for c in df.index.codes])
print("names: ", df.index.names, " nlevels:", df.index.nlevels)
print("get_level_values('quarter'):", list(df.index.get_level_values("quarter")))

east_only = df.loc[["East"]]
print("\nAfter filtering to East only, region level still lists:", list(east_only.index.levels[0]))
print("After remove_unused_levels:", list(east_only.index.remove_unused_levels().levels[0]))

levels: [['East', 'West'], ['Q1', 'Q2', 'Q3']]
codes:  [[0, 0, 0, 1, 1, 1], [0, 1, 2, 0, 1, 2]]
names:  ['region', 'quarter']  nlevels: 2
get_level_values('quarter'): ['Q1', 'Q2', 'Q3', 'Q1', 'Q2', 'Q3']

After filtering to East only, region level still lists: ['East', 'West']
After remove_unused_levels: ['East']


## 05. swaplevel, reorder_levels, sort_index

- swaplevel(i, j) swaps two levels; reorder_levels([...]) sets full order.
- Neither sorts the rows. Call sort_index afterwards, or slicing can fail.
- sort_index(level="quarter") sorts by one level; sort_remaining=False
  leaves the other levels in their current order.

In [6]:
sw = df.swaplevel("region", "quarter")
print("swaplevel (rows NOT re-sorted):\n", sw, sep="")
print("\nswaplevel + sort_index:\n", sw.sort_index(), sep="")
print("\nreorder_levels(['quarter','region']).sort_index().head(3):\n",
      df.reorder_levels(["quarter", "region"]).sort_index().head(3), sep="")
print("\nsort_index(level='quarter', sort_remaining=False).head(4):\n",
      df.sort_index(level="quarter", sort_remaining=False).head(4), sep="")

swaplevel (rows NOT re-sorted):
                sales  cost
quarter region             
Q1      East       18     8
Q2      East       79    36
Q3      East       68    14
Q1      West       49     9
Q2      West       48    28
Q3      West       87    48

swaplevel + sort_index:
                sales  cost
quarter region             
Q1      East       18     8
        West       49     9
Q2      East       79    36
        West       48    28
Q3      East       68    14
        West       87    48

reorder_levels(['quarter','region']).sort_index().head(3):
                sales  cost
quarter region             
Q1      East       18     8
        West       49     9
Q2      East       79    36

sort_index(level='quarter', sort_remaining=False).head(4):
                sales  cost
region quarter             
East   Q1          18     8
West   Q1          49     9
East   Q2          79    36
West   Q2          48    28


## 06. set_index and reset_index

- set_index(["a", "b"]) turns columns into MultiIndex levels.
- reset_index() turns ALL levels back into columns; level="b" only that one.
- reset_index(drop=True) throws the index away instead of keeping it.
- set_index returns a new frame; the original is untouched (no inplace).

In [7]:
flat = pd.DataFrame({"region": ["E", "E", "W"], "quarter": ["Q1", "Q2", "Q1"], "sales": [5, 6, 7]})
mi = flat.set_index(["region", "quarter"])
print("set_index(['region','quarter']):\n", mi, sep="")
print("\nreset_index(level='quarter'):\n", mi.reset_index(level="quarter"), sep="")
print("\nreset_index(drop=True):\n", mi.reset_index(drop=True), sep="")

set_index(['region','quarter']):
                sales
region quarter       
E      Q1           5
       Q2           6
W      Q1           7

reset_index(level='quarter'):
       quarter  sales
region               
E           Q1      5
E           Q2      6
W           Q1      7

reset_index(drop=True):
   sales
0      5
1      6
2      7


## 07. stack and unstack Recap

- unstack moves an index level into columns; stack does the reverse.
- Missing combinations become NaN after unstack; fill_value= fixes that.
- Full details are in 07-Reshaping; here they are tools for MultiIndex.

In [8]:
s = df["sales"]
wide = s.unstack("quarter")
print("unstack('quarter'):\n", wide, sep="")
print("\nunstack(0) (outer level to columns):\n", s.unstack(0), sep="")
print("\nstack() back gives a Series again:", wide.stack().shape)
partial = s.drop(("West", "Q3"))
print("\nMissing combo -> NaN:\n", partial.unstack("quarter"), sep="")
print("\nfill_value=0:\n", partial.unstack("quarter", fill_value=0), sep="")

unstack('quarter'):
quarter  Q1  Q2  Q3
region             
East     18  79  68
West     49  48  87

unstack(0) (outer level to columns):
region   East  West
quarter            
Q1         18    49
Q2         79    48
Q3         68    87

stack() back gives a Series again: (6,)

Missing combo -> NaN:
quarter    Q1    Q2    Q3
region                   
East     18.0  79.0  68.0
West     49.0  48.0   NaN

fill_value=0:
quarter  Q1  Q2  Q3
region             
East     18  79  68
West     49  48   0


## 08. groupby on Index Levels

- groupby(level="region") groups by an index level, no reset_index needed.
- Level can be a name, a number, or a list of them.
- Aggregations like .sum(level=...) were removed; always use groupby.

In [9]:
print("groupby(level='region').sum():\n", df.groupby(level="region").sum(), sep="")
print("\ngroupby(level=['region','quarter']).sum() keeps full detail:", df.groupby(level=[0, 1]).sum().shape)
print("\nMix a level and a column: use pd.Grouper(level=...) or reset_index first")
print(df.groupby([pd.Grouper(level="region"), df["sales"] > 50]).size())

groupby(level='region').sum():
        sales  cost
region             
East      165    58
West      184    85

groupby(level=['region','quarter']).sum() keeps full detail: (6, 2)

Mix a level and a column: use pd.Grouper(level=...) or reset_index first
region  sales
East    False    1
        True     2
West    False    2
        True     1
dtype: int64


## 09. Index Alignment on Arithmetic (Classic Gotcha)

- Arithmetic between Series/DataFrames MATCHES BY LABEL, not by position.
- Labels present in only one side give NaN. This is a silent surprise.
- Fix choices: .add(other, fill_value=0), or use .to_numpy()/.values when
  you truly want position-based math (and lengths must match).
- Duplicate labels make alignment even stranger (see section 11).

In [10]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])
print("a + b (label-matched):\n", a + b, sep="")
print("\na.add(b, fill_value=0):\n", a.add(b, fill_value=0), sep="")
c = pd.Series([100, 200, 300])
print("\nPosition-based (ignoring labels): a.to_numpy() + c.to_numpy() =", a.to_numpy() + c.to_numpy())
print("Same-length but different labels with a + c:\n", a + c, sep="")

d1 = pd.DataFrame({"u": [1, 2], "v": [3, 4]}, index=["r1", "r2"])
d2 = pd.DataFrame({"v": [10, 20], "w": [5, 6]}, index=["r2", "r3"])
print("\nDataFrame + DataFrame aligns BOTH rows and columns:\n", d1 + d2, sep="")

a + b (label-matched):
w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64

a.add(b, fill_value=0):
w    30.0
x     1.0
y    12.0
z    23.0
dtype: float64

Position-based (ignoring labels): a.to_numpy() + c.to_numpy() = [101 202 303]
Same-length but different labels with a + c:
x   NaN
y   NaN
z   NaN
0   NaN
1   NaN
2   NaN
dtype: float64

DataFrame + DataFrame aligns BOTH rows and columns:
     u     v   w
r1 NaN   NaN NaN
r2 NaN  14.0 NaN
r3 NaN   NaN NaN


## 10. reindex, reindex_like, align

- reindex(new_labels) conforms to a new label list: missing -> NaN,
  labels not listed are dropped. It does NOT rename; it re-selects.
- method="ffill"/"bfill" fills gaps only when the index is monotonic.
- reindex_like(other) copies another object's index (and columns).
- align(other, join="outer") gives both objects on a common index.
- GOTCHA: reindex on a duplicate index raises ValueError.

In [11]:
r = pd.Series([1.0, 2.0, 3.0], index=[10, 20, 30])
print("reindex([10, 15, 30, 40]):\n", r.reindex([10, 15, 30, 40]), sep="")
print("\nreindex with method='ffill':\n", r.reindex([10, 15, 30, 40], method="ffill"), sep="")
print("\nreindex with fill_value=0:", r.reindex([10, 15], fill_value=0).tolist())

other = pd.Series([0, 0, 0, 0], index=[30, 20, 40, 10])
print("\nreindex_like(other).index:", r.reindex_like(other).index.tolist())

left, right = r.align(other, join="outer")
print("align outer -> left:", left.tolist(), " right:", right.tolist())
left, right = r.align(other, join="inner")
print("align inner -> shared labels:", left.index.tolist())

try:
    pd.Series([1, 2], index=["a", "a"]).reindex(["a", "b"])
except ValueError as e:
    print("reindex with duplicate index ->", e)

reindex([10, 15, 30, 40]):
10    1.0
15    NaN
30    3.0
40    NaN
dtype: float64

reindex with method='ffill':
10    1.0
15    1.0
30    3.0
40    3.0
dtype: float64

reindex with fill_value=0: [1.0, 0.0]

reindex_like(other).index: [30, 20, 40, 10]
align outer -> left: [1.0, 2.0, 3.0, nan]  right: [0, 0, 0, 0]
align inner -> shared labels: [10, 20, 30]
reindex with duplicate index -> cannot reindex on an axis with duplicate labels


## 11. Duplicate Index Problems

- pandas allows duplicate labels. That is useful sometimes, risky mostly.
- .loc["a"] returns a Series/DataFrame (many rows) instead of one value,
  so the return TYPE depends on the data: a nasty source of bugs.
- Check with index.is_unique and index.duplicated().
- flags.allows_duplicate_labels = False makes pandas raise on duplicates.
- Fix: reset_index, groupby-aggregate duplicates, or de-dupe on purpose.

In [12]:
dup = pd.Series([1, 2, 3], index=["a", "a", "b"])
print("dup.index.is_unique:", dup.index.is_unique)
print("dup.index.duplicated():", dup.index.duplicated().tolist())
print("dup.loc['a'] returns:\n", dup.loc["a"], sep="")
print("dup.loc['b'] returns a scalar:", dup.loc["b"])
print("Keep first of each label:", dup[~dup.index.duplicated(keep="first")].tolist())
print("Aggregate duplicates:", dup.groupby(level=0).sum().to_dict())

strict = pd.Series([1, 2], index=["a", "b"]).set_flags(allows_duplicate_labels=False)
try:
    strict.reindex(["a", "a"])
except Exception as e:
    print("Strict flag blocked it ->", type(e).__name__)

dup.index.is_unique: False
dup.index.duplicated(): [False, True, False]
dup.loc['a'] returns:
a    1
a    2
dtype: int64
dup.loc['b'] returns a scalar: 3
Keep first of each label: [1, 3]
Aggregate duplicates: {'a': 3, 'b': 3}
Strict flag blocked it -> DuplicateLabelError


## 12. RangeIndex

- The default index is RangeIndex(start, stop, step): stored as 3 numbers,
  not an array, so it costs almost no memory.
- After filtering rows the labels stay (e.g. 1, 3, 4) but it is no
  longer a RangeIndex; it becomes Index of int64.
- reset_index(drop=True) builds a fresh RangeIndex.
- GOTCHA: .loc[3] is a LABEL, .iloc[3] is a POSITION; after filtering
  they refer to different rows.

In [13]:
base = pd.DataFrame({"v": [10, 20, 30, 40, 50]})
print("default index:", base.index)
filt = base[base["v"] > 20]
print("after filter:", filt.index.tolist(), "type:", type(filt.index).__name__)
print("filt.loc[3] ->", filt.loc[3, "v"], "  filt.iloc[0] ->", filt.iloc[0]["v"])
print("after reset_index(drop=True):", filt.reset_index(drop=True).index)
print("RangeIndex memory (bytes):", base.index.memory_usage(), " vs Index(int64) of 5:", pd.Index([0, 1, 2, 3, 4]).memory_usage())

default index: RangeIndex(start=0, stop=5, step=1)
after filter: [2, 3, 4] type: Index
filt.loc[3] -> 40   filt.iloc[0] -> 30
after reset_index(drop=True): RangeIndex(start=0, stop=3, step=1)
RangeIndex memory (bytes): 132  vs Index(int64) of 5: 40


## 13. IntervalIndex, pd.cut, pd.qcut

- pd.cut(values, bins) makes fixed-width/custom bins; result dtype is
  category with Interval labels. Intervals are right-closed by default:
  (0, 10]. right=False gives [0, 10). include_lowest=True keeps the min.
- pd.qcut(values, q) makes bins with (about) equal COUNTS (quantiles).
- labels=[...] replaces interval text with names.
- pd.IntervalIndex.from_breaks and .get_indexer / .contains do lookups.
- GOTCHA: a value exactly equal to the lowest edge is NaN under
  right-closed bins unless include_lowest=True.

In [14]:
ages = pd.Series([0, 5, 18, 25, 40, 65, 90])
cut = pd.cut(ages, bins=[0, 18, 40, 100])
print("pd.cut with edges [0, 18, 40, 100]:\n", pd.DataFrame({"age": ages, "bin": cut}), sep="")
print("\ninclude_lowest=True:", pd.cut(ages, [0, 18, 40, 100], include_lowest=True).astype(str).tolist()[:2])
print("labels:", pd.cut(ages, [-1, 18, 40, 100], labels=["child", "adult", "senior"]).tolist())
print("right=False:", pd.cut(ages, [0, 18, 40, 100], right=False).astype(str).tolist())

scores = pd.Series(np.arange(1, 13))
q = pd.qcut(scores, 4, labels=["Q1", "Q2", "Q3", "Q4"])
print("\nqcut into 4 equal-count groups:", q.value_counts().sort_index().to_dict())

ii = pd.IntervalIndex.from_breaks([0, 10, 20, 30])
print("\nIntervalIndex:", ii.tolist())
print("12 falls in bucket:", ii.get_indexer([12]).tolist(), "-> ", ii[ii.get_indexer([12])[0]])
print("Series indexed by intervals, lookup of 25:", pd.Series(["low", "mid", "high"], index=ii).loc[25])

pd.cut with edges [0, 18, 40, 100]:
   age            bin
0    0            NaN
1    5    (0.0, 18.0]
2   18    (0.0, 18.0]
3   25   (18.0, 40.0]
4   40   (18.0, 40.0]
5   65  (40.0, 100.0]
6   90  (40.0, 100.0]

include_lowest=True: ['(-0.001, 18.0]', '(-0.001, 18.0]']
labels: ['child', 'child', 'child', 'adult', 'adult', 'senior', 'senior']
right=False: ['[0, 18)', '[0, 18)', '[18, 40)', '[18, 40)', '[40, 100)', '[40, 100)', '[40, 100)']

qcut into 4 equal-count groups: {'Q1': 3, 'Q2': 3, 'Q3': 3, 'Q4': 3}

IntervalIndex: [Interval(0, 10, closed='right'), Interval(10, 20, closed='right'), Interval(20, 30, closed='right')]
12 falls in bucket: [1] ->  (10, 20]
Series indexed by intervals, lookup of 25: high


## 14. isin and between

- s.isin([...]) -> boolean mask: "value is one of these".
- s.between(lo, hi) -> mask; inclusive="both"/"neither"/"left"/"right".
- ~mask negates. Prefer isin over many `==` joined with `|`.
- Index has isin too: df[df.index.isin([...])].
- NaN is never "between" anything and isin([np.nan]) matches NaN.

In [15]:
city = pd.Series(["Delhi", "Pune", "Goa", "Pune", None, "Agra"])
print("isin:", city.isin(["Pune", "Goa"]).tolist())
print("~isin:", (~city.isin(["Pune", "Goa"])).tolist())
n = pd.Series([1, 5, 10, np.nan, 15])
print("between(5, 10):", n.between(5, 10).tolist())
print("between(5, 10, inclusive='neither'):", n.between(5, 10, inclusive="neither").tolist())
print("isin([np.nan]) matches NaN:", n.isin([np.nan]).tolist())
print("index.isin:", df[df.index.get_level_values("quarter").isin(["Q1"])].shape)

isin: [False, True, True, True, False, False]
~isin: [True, False, False, False, True, True]
between(5, 10): [False, True, True, False, False]
between(5, 10, inclusive='neither'): [False, False, False, False, False]
isin([np.nan]) matches NaN: [False, False, False, True, False]
index.isin: (2, 2)


## 15. query and eval

- df.query("a > 5 and b == 'x'") filters with a string expression: short,
  chainable, no repeated df[...] names.
- Use @var to refer to a Python variable; backticks for odd column names.
- Index levels can be used by name inside query.
- df.eval("c = a + b") builds new columns; with numexpr installed, big
  frames can be faster. For small data it is about readability, not speed.
- GOTCHA: query is a mini-language, not full Python. Not every function works.

In [16]:
people = pd.DataFrame({"name": ["A", "B", "C", "D"], "age": [20, 35, 50, 41], "pay k": [30, 55, 90, 70]})
limit = 30
print("query('age > @limit and `pay k` < 80'):\n", people.query("age > @limit and `pay k` < 80"), sep="")
print("\nquery on index level 'region':\n", df.query("region == 'East' and sales > 0").shape, sep="")
print("\nname in list:", people.query("name in ['A', 'D']")["name"].tolist())
print("\neval creates a column (returns a new frame):\n", people.eval("pay_per_age = `pay k` / age").round(2), sep="")

query('age > @limit and `pay k` < 80'):
  name  age  pay k
1    B   35     55
3    D   41     70

query on index level 'region':
(3, 2)

name in list: ['A', 'D']

eval creates a column (returns a new frame):
  name  age  pay k  pay_per_age
0    A   20     30         1.50
1    B   35     55         1.57
2    C   50     90         1.80
3    D   41     70         1.71


## 16. where and mask

- s.where(cond, other): KEEP where cond is True, REPLACE where False.
- s.mask(cond, other): the opposite: REPLACE where cond is True.
- Same shape is returned (rows are not dropped, unlike boolean indexing).
- Default replacement is NaN, which turns ints into floats.
- np.where(cond, a, b) is the fast numpy version returning an array.

In [17]:
v = pd.Series([5, -3, 8, -1, 0])
print("where(v > 0):", v.where(v > 0).tolist())
print("where(v > 0, 0):", v.where(v > 0, 0).tolist())
print("mask(v < 0, 99):", v.mask(v < 0, 99).tolist())
print("np.where label:", np.where(v > 0, "pos", "non-pos").tolist())
print("DataFrame where:\n", pd.DataFrame({"a": [1, -2], "b": [-3, 4]}).where(lambda d: d > 0, other=0), sep="")

where(v > 0): [5.0, nan, 8.0, nan, nan]
where(v > 0, 0): [5, 0, 8, 0, 0]
mask(v < 0, 99): [5, 99, 8, 99, 0]
np.where label: ['pos', 'non-pos', 'pos', 'non-pos', 'non-pos']
DataFrame where:
   a  b
0  1  0
1  0  4


## 17. .filter - Select Labels by Pattern

- df.filter(items=[...]) picks labels that exist (silently skips missing).
- df.filter(like="sub") keeps labels containing the substring.
- df.filter(regex="^q") keeps labels matching a regex.
- axis=0 filters ROW labels, axis=1 (default for DataFrame) filters columns.
- It filters LABELS, not values; for values use boolean masks or query.

In [18]:
cols = pd.DataFrame(np.arange(6).reshape(2, 3), columns=["q1_sales", "q2_sales", "total_cost"], index=["row_a", "row_b"])
print("filter(like='sales'):", cols.filter(like="sales").columns.tolist())
print("filter(regex='^q'):", cols.filter(regex="^q").columns.tolist())
print("filter(items=['total_cost', 'nope']):", cols.filter(items=["total_cost", "nope"]).columns.tolist())
print("filter(like='_b', axis=0):", cols.filter(like="_b", axis=0).index.tolist())

filter(like='sales'): ['q1_sales', 'q2_sales']
filter(regex='^q'): ['q1_sales', 'q2_sales']
filter(items=['total_cost', 'nope']): ['total_cost']
filter(like='_b', axis=0): ['row_b']


## 18. Boolean Masks Combined With Index Levels

- Use & (and), | (or), ~ (not) with PARENTHESES around each comparison;
  Python's `and`/`or` raise "truth value is ambiguous".
- Index-level conditions: get_level_values gives an Index you can compare.
- Combine: mask = (level == x) & (column > y).
- GOTCHA: precedence! `a > 1 & b < 2` is parsed wrong without brackets.

In [19]:
lvl = df.index.get_level_values("quarter")
mask = (lvl != "Q2") & (df["sales"] > df["sales"].median())
print("Rows not in Q2 with above-median sales:\n", df[mask], sep="")
try:
    df[(df["sales"] > 1) and (df["cost"] > 1)]
except ValueError as e:
    print("\nUsing `and` ->", type(e).__name__)

print("\n# End of MultiIndex and Advanced Indexing")

Rows not in Q2 with above-median sales:
                sales  cost
region quarter             
East   Q3          68    14
West   Q3          87    48

Using `and` -> ValueError

# End of MultiIndex and Advanced Indexing
